In [18]:
import sys
import os
os.environ['PROJ_DATA'] = "/pscratch/sd/p/plutzner/proj_data"
import xarray as xr
import torch
import torchinfo
import random
import numpy as np
import importlib as imp
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import cartopy.crs as ccrs
import json
import pickle
import gzip
import scipy
from scipy import stats
from cftime import DatetimeNoLeap
from datetime import datetime
from sklearn.metrics import mean_squared_error
#import matplotlib.colors as mcolorsxx

# %load_ext autoreload
# %autoreload 2
import utils
import utils.filemethods as filemethods
import databuilder.data_loader as data_loader
from utils.filemethods import open_data_file
from utils import utils


"""
(1) Open 0101 and 0151 data files containing Z500, PRECT, TS
(2) Check the files for correct magnitudes and variables
(3) Shift 0101 data by -60225 days to 1685-1849
(4) Merge the two datasets into a single dataset with continuous time (1685 - 2014)
(5) Save the merged dataset to a new NetCDF file
(6) Print the shape of the merged dataset
(7) Print the time range of the merged dataset
(8) Print the variable names in the merged dataset

"""
    

'\n(1) Open 0101 and 0151 data files containing Z500, PRECT, TS\n(2) Check the files for correct magnitudes and variables\n(3) Shift 0101 data by -60225 days to 1685-1849\n(4) Merge the two datasets into a single dataset with continuous time (1685 - 2014)\n(5) Save the merged dataset to a new NetCDF file\n(6) Print the shape of the merged dataset\n(7) Print the time range of the merged dataset\n(8) Print the variable names in the merged dataset\n\n'

In [19]:
def merge_input_files(training_file, val_file, output_file):
    # Open the training data file
    ds_train = open_data_file(training_file)
    ds_train_input = ds_train['x']

    # Select only 1850-2014 (not 2015)
    ds_train_input = ds_train_input.sel(time=slice('1850-01-01', '2014-12-31'))

    # Open the validation data file
    ds_val = open_data_file(val_file)
    ds_val_input = ds_val['x']

    # Shift the time in the test dataset by -60225 days
    # Datetime no leap is used to avoid leap years in the time series
    print("Time range of train dataset before shift:", ds_train_input['time'].values.min(), "to", ds_train_input['time'].values.max())
    ds_train_input['time'] = xr.cftime_range(start='1685-01-01', 
                                       end='1849-12-31', 
                                       freq='D',
                                       calendar='noleap')

    print("Time range of train dataset after shift:", ds_train_input['time'].values.min(), "to", ds_train_input['time'].values.max())

    print(f"ds_train_input: {ds_train_input}")

    # Data is in the form [time, lat, lon, channel] in ['x'], and [time] in ['y']
    # Merge the two datasets along the time dimension
    ds_merged_training = xr.merge([ds_train_input, ds_val_input], compat='override')
    
    print(f"ds_merged_training: {ds_merged_training}")

    # # Assign the merged time coordinate to the merged dataset
    # ds_merged_training = ds_merged_training.assign_coords(time=merged_time) ? 

    # shift and merge target too: 
    ds_train_target = ds_train['y']
    ds_val_target = ds_val['y']

    ds_train_target['time'] = ds_train_input['time']

    ds_merged_training_target =  xr.merge([ds_train_target, ds_val_target], compat='override')
    print(f"ds_merged_training_target: {ds_merged_training_target}")
    print(f"ds_merged_Training_target time: {ds_merged_training_target.time}")

    # Save the merged dataset to a new pkl file maintingin the ['x'] and ['y'] dictionary structure

    d_train_new = {
        'x': ds_merged_training,
        'y': ds_merged_training_target
    }

    if np.any(np.isnan(d_train_new['x'])):
        print("Input data has nans")
        # print location of nans
        nan_locations = np.where(np.isnan(d_train_new['x']))
        print(nan_locations)

In [20]:
train_fn = '/pscratch/sd/p/plutzner/E3SM/bigdata/presaved/exp140_trimmed_train_dat.nc'
val_fn = '/pscratch/sd/p/plutzner/E3SM/bigdata/presaved/exp140_trimmed_val_dat.nc'
test_fn = '/pscratch/sd/p/plutzner/E3SM/bigdata/presaved/exp140_trimmed_test_dat.nc'

output_fn = '/pscratch/sd/p/plutzner/E3SM/bigdata/presaved/exp140_d_train_merged.pkl'
merge_input_files(train_fn, val_fn, output_fn)

Time range of train dataset before shift: 1850-02-12 00:00:00 to 2014-12-17 00:00:00


ValueError: conflicting sizes for dimension 'time': length 29974 on <this-array> and length 60225 on {'lat': 'lat', 'lon': 'lon', 'channel': 'channel', 'time': 'time'}

In [5]:
test_fn = '/pscratch/sd/p/plutzner/E3SM/bigdata/presaved/exp140_d_test.pkl'

# Split testing dataset into two halves: 
new_validation_dates = [1850, 1931]
new_test_dates = [1932, 2014]

# load testing dataset
original_testing = open_data_file(test_fn)

# Select the new validation and test periods
new_val_input = original_testing['x'].sel(time=slice(f"{new_validation_dates[0]}-01-01", f"{new_validation_dates[1]}-12-31"))
new_val_target = original_testing['y'].sel(time=slice(f"{new_validation_dates[0]}-01-01", f"{new_validation_dates[1]}-12-31"))
new_test_input = original_testing['x'].sel(time=slice(f"{new_test_dates[0]}-01-01", f"{new_test_dates[1]}-12-31"))
new_test_target = original_testing['y'].sel(time=slice(f"{new_test_dates[0]}-01-01", f"{new_test_dates[1]}-12-31"))

# save the new validation and test datasets as pkl
new_val_data = {
    'x': new_val_input,
    'y': new_val_target
}
new_test_data = {
    'x': new_test_input,
    'y': new_test_target
}
new_val_fn = '/pscratch/sd/p/plutzner/E3SM/bigdata/presaved/exp140_d_val_modified.pkl'
new_test_fn = '/pscratch/sd/p/plutzner/E3SM/bigdata/presaved/exp140_d_test_modified.pkl'

with open(new_val_fn, 'wb') as f:
    pickle.dump(new_val_data, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(new_test_fn, 'wb') as f:
    pickle.dump(new_test_data, f, protocol=pickle.HIGHEST_PROTOCOL)

### Checking inputs for nans - none

In [21]:
exp140 = open_data_file('/pscratch/sd/p/plutzner/E3SM/bigdata/presaved/exp144_trimmed_train_dat.nc')

In [22]:
if np.any(np.isnan(exp140['x'])):
    print("Input data has nans")
    # print location of nans
    nan_locations = np.where(np.isnan(exp140['x']))
    print(nan_locations)

In [23]:
exp140_val = open_data_file('/pscratch/sd/p/plutzner/E3SM/bigdata/presaved/exp144_trimmed_val_dat.nc')

if np.any(np.isnan(exp140_val['x'])):
    print("Input data has nans")
    # print location of nans
    nan_locations = np.where(np.isnan(exp140_val['x']))
    print(nan_locations)

In [24]:
exp140_test = open_data_file('/pscratch/sd/p/plutzner/E3SM/bigdata/presaved/exp144_trimmed_test_dat.nc')

if np.any(np.isnan(exp140_test['x'])):
    print("Input data has nans")
    # print location of nans
    nan_locations = np.where(np.isnan(exp140_test['x']))
    print(nan_locations)

In [26]:
exp140

<xarray.Dataset> Size: 8GB
Dimensions:        (time: 10711, lat: 181, lon: 360, channel: 3)
Coordinates:
  * time           (time) datetime64[ns] 86kB 1940-01-29 ... 1998-12-17
    isobaricInhPa  float64 8B ...
  * lat            (lat) float64 1kB 90.0 89.0 88.0 87.0 ... -88.0 -89.0 -90.0
  * lon            (lon) float64 3kB 0.0 1.0 2.0 3.0 ... 356.0 357.0 358.0 359.0
  * channel        (channel) int64 24B 0 1 2
Data variables:
    x              (time, lat, lon, channel) float32 8GB -0.149 2.446 ... 83.01
    y              (time) float64 86kB ...

In [28]:
exp140_shifted = open_data_file('/pscratch/sd/p/plutzner/E3SM/bigdata/presaved/exp140_trimmed_train_dat_shifted.nc')
exp140_shifted.time
exp140_shifted

<xarray.Dataset> Size: 31GB
Dimensions:  (time: 29974, lon: 360, lat: 180, channel: 2)
Coordinates:
  * time     (time) object 240kB 1685-02-12 00:00:00 ... 1849-12-17 00:00:00
  * lon      (lon) float64 3kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * lat      (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * channel  (channel) float64 16B 0.0 1.0
Data variables:
    x        (time, lat, lon, channel) float64 31GB ...
    y        (time) float64 240kB ...
Attributes:
    CDI:          Climate Data Interface version 2.4.4 (https://mpimet.mpg.de...
    Conventions:  CF-1.6
    history:      Fri Jul 25 13:06:35 2025: cdo shifttime,-60225days exp140_t...
    CDO:          Climate Data Operators version 2.4.4 (https://mpimet.mpg.de...